# TP - DLA - Privacy defense (III) - use of OPACUS

Small example of the use of OPACUS for learning (under DP mechanism) a deep model

In [1]:
# ======================================================
# Differentially Private Deep Learning in ~20 lines
# ======================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from opacus import PrivacyEngine

# -------------------------------
# 1. Data
# -------------------------------
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)

# -------------------------------
# 2. Model
# -------------------------------
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = Net()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# -------------------------------
# 3. Make optimizer private
# -------------------------------
privacy_engine = PrivacyEngine()
model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=1.2,   # adjust for stronger/weaker DP
    max_grad_norm=1.0,
)

# -------------------------------
# 4. Training loop (1 epoch demo)
# -------------------------------
model.train()
for batch_idx, (data, target) in enumerate(train_loader):
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    if batch_idx % 200 == 0:
        print(f"Train step {batch_idx}, Loss: {loss.item():.4f}")

# -------------------------------
# 5. Report privacy budget
# -------------------------------
epsilon = privacy_engine.get_epsilon(delta=1e-5)
print(f"Final privacy budget: ε = {epsilon:.2f}, δ = 1e-5")


/home/souna/Bureau/Master2/DLA/tp-uv/.venv/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
/tmp/ipykernel_11309/2887922156.py:57: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Train step 0, Loss: 2.3049
Train step 200, Loss: 1.1849
Train step 400, Loss: 1.2398
Train step 600, Loss: 0.5198
Train step 800, Loss: 0.5283
Train step 1000, Loss: 0.6168
Train step 1200, Loss: 0.8170
Train step 1400, Loss: 1.2053
Train step 1600, Loss: 1.3285
Train step 1800, Loss: 0.8726
Train step 2000, Loss: 0.9416
Train step 2200, Loss: 1.4482
Train step 2400, Loss: 0.2688
Train step 2600, Loss: 0.2484
Train step 2800, Loss: 1.0701
Final privacy budget: ε = 0.07, δ = 1e-5
